# P4 Agent and Tools — Implementation

This notebook sets up the P4 package, runs the unit tests, validates the independent tools, and provides the integration points for P2 and P3.

In [1]:
!pip -q install langgraph langchain-core pydantic python-docx pytest

## 1. Upload and extract the P4 project files

Upload `P4_Project_Files.zip` from your working directory.

In [2]:
from google.colab import files
uploaded = files.upload()

import os
import zipfile

archive_path = next(iter(uploaded))
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall('/content')

os.chdir('/content/P4_Project_Files')
print('Working directory:', os.getcwd())


Saving P4_Project_Files.zip to P4_Project_Files.zip
Working directory: /content/P4_Project_Files


## 2. Run unit tests

In [3]:
!PYTHONPATH=. pytest -q

......                                                                   [100%]
6 passed in 0.03s


## 3. Validate independent tools

In [4]:
from agent.tools.calculator import calculator
from agent.tools.unit_convert import unit_convert
from agent.tools.company_lookup import company_lookup
from agent.tools.metric_lookup import metric_lookup
from agent.guards import gatekeeper

print(calculator('(250 - 50) / 2'))
print(unit_convert(250, 'lakh', 'crore'))
print(company_lookup('HUL'))
print(metric_lookup('A&P'))
print(gatekeeper('Should I buy Nestle India shares?'))


{'expression': '(250 - 50) / 2', 'value': 100.0}
{'input_value': 250, 'from_unit': 'lakh', 'value': 2.5, 'to_unit': 'crore'}
{'status': 'supported', 'input': 'HUL', 'canonical': 'Hindustan Unilever', 'aliases': ['HUL', 'Hindustan Unilever']}
{'status': 'matched', 'canonical': 'a&p', 'synonyms': ['advertisement and sales promotion', 'advertising and publicity']}
{'status': 'refused', 'refusal_code': 'prohibited_investment_advice', 'message': 'I can help with historical, source-grounded company research, but not investment advice, forecasts or valuations.'}


## 4. Configure P3 retrieval adapters

Replace the function bodies below with imports/calls to P3's implementation after confirming the exact signatures and response fields.

In [5]:
from agent.tools.retrieval import RetrievalAdapter

def p3_retrieve(query: str, k: int = 5, filters: dict | None = None):
    raise NotImplementedError('Connect the confirmed P3 retrieval interface.')

def p3_get_table(source_id: str, page: int | None = None, table_id: str | None = None):
    raise NotImplementedError('Connect the confirmed P3 table extraction interface.')

retrieval = RetrievalAdapter(
    retrieve_fn=p3_retrieve,
    table_fn=p3_get_table
)

# After connecting P3:
# evidence = retrieval.search_reports(
#     query='Dabur FY24 revenue from operations',
#     k=5,
#     filters={'company': 'Dabur India', 'fiscal_year': 'FY24'}
# )
# print(evidence)


## 5. Configure P2 model and prompt functions

Use the shared model client supplied by P2. Keep model calls within that client so that common runtime settings, retries, and token accounting are retained.

In [6]:
def planner_fn(user_query: str, context: dict) -> dict:
    raise NotImplementedError('Connect P2 planner prompt and shared client.')

def writer_fn(user_query: str, observations: list, context: dict) -> dict:
    raise NotImplementedError('Connect P2 writer prompt and shared client.')

def critic_fn(draft: dict, observations: list) -> dict:
    raise NotImplementedError('Connect P2 critic prompt and shared client.')


## 6. Build and invoke the graph

Run this section after the P2 and P3 adapters have been connected and their return structures have been validated.

In [7]:
from agent.tools.registry import build_tool_registry
from agent.graph import build_graph

# tools = build_tool_registry(retrieval)
# app = build_graph(tools, planner_fn, writer_fn, critic_fn)
#
# result = app.invoke({
#     'user_query': 'What was Dabur revenue from operations in FY24?',
#     'context': {},
#     'token_budget': 6000,
#     'approved': False
# })
# print(result.get('final'))


## 7. Verify the report approval gate

In [8]:
from agent.tools.report import generate_report

# Export must remain blocked until a human approves the preview.
print(generate_report(
    title='Competitive Research Brief',
    sections=[],
    approved=False
))


{'status': 'approval_required', 'message': 'A human must approve the preview before export.'}


## 8. Integration checklist

- [ ] Unit tests pass.
- [ ] P2 model client and prompt interfaces confirmed.
- [ ] P3 retrieval and table-extraction interfaces confirmed.
- [ ] `search_reports` and `get_table` return real source records.
- [ ] Source IDs, document titles, and page numbers are traceable.
- [ ] One in-scope historical lookup completes with evidence.
- [ ] Refusal and missing-evidence paths are tested.
- [ ] Response validates against the P1 schema.
- [ ] P5 tracing metadata is connected.
- [ ] Export requires explicit human approval.